🗺️🌍 uRBAN

O objetivo desse agente e servir como um consultor urbano digital para projetos em Belo Horizonte.
A ideia e simples: voce tem um arquivo IFC de um projeto e quer saber se ele esta dentro das regras urbanisticas da cidade. Em vez de precisar consultar manualmente a legislacao, abrir os mapas da prefeitura e cruzar as informacoes na mao, o agente faz tudo isso automaticamente.
Voce so digita uma pergunta em linguagem natural, o agente le o IFC, consulta os dados reais da PBH e te devolve um laudo tecnico.
Os dados urbanos que escolhemos sao um pacote enxuto pra testar o sistema. O potencial aqui e muito maior, da pra expandir com muito mais camadas de dados da prefeitura e responder perguntas cada vez mais complexas sobre qualquer projeto na cidade.

1. Instalacao das Bibliotecas

Intalamos as 4 bibliotecas:
- langgraph + langchain + langchain-anthropic: que serao usados para o framework dos agentes
- anthropic: que da acesso a API do Claude
- geopandas + shapely + pyproj: que serao necessarios para consultas geoespaciais nos Shapefiles da PBH
- ifcopenshell:que fara a leitura e extracao de dados de arquivos .ifc (BIM)


In [ ]:
!pip install langgraph langchain langchain-anthropic anthropic \
             geopandas shapely pyproj pandas requests -q
!pip install ifcopenshell -q
print(" ✅ Instalacao check!")

 ✅ Instalacao check!


2. Ativando a chave API

Configuramos o sistema para usar o Claude da Anthropic como LLM para interpretar os dados e gerar os laudos em linguagem natural.
A chave e carregada via Colab Secrets para nao ficar exposta no codigo e incorporamos tbm um fallback para input manual caso os secrets nao estejam configurados.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")   #Aqui configurei a API_KEY como fizemos na aula.
    print("✅ API Key carregada via Colab Secrets")
except ImportError:
    if "ANTHROPIC_API_KEY" in os.environ:
        print("✅ API Key encontrada nas variáveis de ambiente")
    else:
        import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Cole sua ANTHROPIC_API_KEY: ")
        print("✅ API Key configurada manualmente")

✅ API Key carregada via Colab Secrets


3. Download do IFC

Professor, por falta de material e tempo, mas com o intuito de criar um agente mais focado em Urbanismo nos modificamos o seu IFC (palmeiras_142) mantendo todos os atributos do projeto alterando apenas as coordenadas. Por que isso? Estamos desenvolvendo nosso TFM em um recorte de BH e ja temos conhecimento do vasto bando de dados que a cidade possui. A ideia do agente que criamos aqui e ser um consultor capaz de trazer como output se o Projeto atende as conformidades urbanas do zoneamento local que foram fornecidas ao agente para realizar as contultas.

Usamos o gdown para baixar o arquivo IFC direto do drive como aprendemos em aula.

In [ ]:
!gdown 1UqY3FXceVHn6ndLe9M_wMcTNaiQox57p #Id do Arquivo hospedado no drive

Downloading...
From: https://drive.google.com/uc?id=1UqY3FXceVHn6ndLe9M_wMcTNaiQox57p
To: /content/espirito_santo_507.ifc
100% 50.5k/50.5k [00:00<00:00, 41.4MB/s]


4. Verificando se o IFC subiu corretamente

Antes de rodar o sistema, verificamos se o IFC esta valido, com  as informacoes necessarias e com as coordenadas atualizadas depois que alteramos o seu IFC:
- IfcSite: localizacao georreferenciada (lat/lon)
- IfcBuilding: uso da edificacao
- IfcBuildingStorey: numero de pavimentos
- IfcSpace: ambientes para calculo de area


In [ ]:
import ifcopenshell

modelo = ifcopenshell.open("/content/espirito_santo_507.ifc")

sites     = modelo.by_type("IfcSite")
buildings = modelo.by_type("IfcBuilding")
pavs      = modelo.by_type("IfcBuildingStorey")
espacos   = modelo.by_type("IfcSpace")

print("Site:       ", sites[0].Name if sites else "sem site")
print("Edificio:   ", buildings[0].Name if buildings else "sem edificio")
print("Pavimentos: ", len(pavs))
print("Espacos:    ", len(espacos))
print("Lat:        ", sites[0].RefLatitude if sites else "sem coordenada")
print("Lon:        ", sites[0].RefLongitude if sites else "sem coordenada")

Site:        Rua Espirito Santo, 507 — Belo Horizonte/MG
Edificio:    Residência Unifamiliar — Palmeiras 142
Pavimentos:  2
Espacos:     12
Lat:         (-19, 55, 40, 0)
Lon:         (-43, 56, 22, 0)



5. Imports principais e inicializacao do LLM

Carregamos todos os modulos necessarios para o sistema:
- TypedDict: para definIR o contrato de estado compartilhado entre os agentes
- StateGraph, START, END: que sao os blocos de construcao do LangGraph
- ChatAnthropic:que e a interface com o modelo Claude
- geopandas + shapely: para fazer as consultas espaciais nos Shapefiles
- ifcopenshell: que vai ler do arquivo IFC


In [ ]:
import json
import random
from typing import TypedDict, Annotated, List, Optional

# LangChain
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# GeoPandas e IfcOpenShell
import geopandas as gpd
from shapely.geometry import Point
import ifcopenshell
import ifcopenshell.util.element as ifc_util

# LLM padrão
llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0, max_tokens=1024) #Configuramos o temperature=0 para diminuir a possibilidade de alucinacao do modelo.
                                                                                      #Mas sabemos que mesmo "amarrando" o modelo, isso nao da 100% de certeza sobre o output gerado.

print("✅ Imports e LLM prontos!")
print("✅ LangGraph importado com sucesso")

✅ Imports e LLM prontos!
✅ LangGraph importado com sucesso


6. Importando com os arquivos que serao usado para a consulta do meu Agente

Professor, como comentado anteriormente a ideia do nosso agente e ele poder servir como um consultor urbano. E ao se tratar de dados urbanos sabemos que nao vamos ler apenas numeros pois o agente vai precisar fazer consultas geometrias vetoriais (polígonos, linhas, pontos) com atributos associados.
Por isso escolhemos subir os arquivos Shapefiles forneidos pelo bando de dados da prefeitura para depois usando o geopanda realizar a leitura dos agentes durante a análise. Os arquivos eles nao treinam o modelo, apenas servem como base de consulta geográfica em tempo real.

Conforme fomos trabalhando nos codigos descobrimos a possibilidade de usar o WFS direto. Ele seria o ideal para nao precisar baixar nada e ate mesmo ampliar as possibilidades de dados analisados, com isso o codigo consultaria a API da PBH em tempo real mas o servidor da PBH bloqueou requisicoes de IPs externos como os do Colab com erro 403 :( isso foi bem triste!


obs. Os dados urbanos escolhidos foi um pacote minimo para que testar o agente. Isso aqui tem um potencial muito maior.

In [60]:
import os
import zipfile

os.makedirs("/content/bhgeo", exist_ok=True)

shapefiles = {
    "ADE_11181.zip":                "1J3bSCuEVA7Nu_oV-xRLlX84CfRsqoKEK",
    "ADE_SETORES_11181.zip":        "1AyUTggNb4XBgwOW7hxr2LFmnvlTG4_X6",
    "CLASSIFICACAO_VIARIA_11181.zip": "1yPRn726W9FiZGDlJp2Gy_I1k1g51vO3T",
    "ZONEAMENTO_11181.zip":         "1u2Mx9yHnEk09ZW10YxdXZcFKkL8FUmLX",
}

for nome, id_drive in shapefiles.items():
    print("Baixando " + nome + "...")
    os.system("gdown " + id_drive)
    with zipfile.ZipFile(nome, 'r') as z:
        z.extractall("/content/bhgeo")
    print("✅ " + nome + " extraído")

print("\nArquivos disponíveis em /content/bhgeo:")
for f in sorted(os.listdir("/content/bhgeo")):
    print("  " + f)

Baixando ADE_11181.zip...
✅ ADE_11181.zip extraído
Baixando ADE_SETORES_11181.zip...
✅ ADE_SETORES_11181.zip extraído
Baixando CLASSIFICACAO_VIARIA_11181.zip...
✅ CLASSIFICACAO_VIARIA_11181.zip extraído
Baixando ZONEAMENTO_11181.zip...
✅ ZONEAMENTO_11181.zip extraído

Arquivos disponíveis em /content/bhgeo:
  ADE_11181.cst
  ADE_11181.dbf
  ADE_11181.prj
  ADE_11181.shp
  ADE_11181.shx
  ADE_SETORES_11181.cst
  ADE_SETORES_11181.dbf
  ADE_SETORES_11181.prj
  ADE_SETORES_11181.shp
  ADE_SETORES_11181.shx
  CLASSIFICACAO_VIARIA_11181.cst
  CLASSIFICACAO_VIARIA_11181.dbf
  CLASSIFICACAO_VIARIA_11181.prj
  CLASSIFICACAO_VIARIA_11181.shp
  CLASSIFICACAO_VIARIA_11181.shx
  ZONEAMENTO_11181.cst
  ZONEAMENTO_11181.dbf
  ZONEAMENTO_11181.prj
  ZONEAMENTO_11181.shp
  ZONEAMENTO_11181.shx


7. Leitura e inspecao dos dados para ver se tudo esta sendo lido corretamente
- Abre cada shapefile
- Converte o sistema de coordenadas
- Imprime as colunas e o primeiro registro

Por que isso e necessario?
Os Shapefiles da PBH tem nomes de colunas que so a PBH sabe quais sao. Por exemplo, a coluna que guarda a sigla da zona pode se chamar SIGLA, ZONA, COD_ZONA, NM_ZONA e nao temos como saber sem abrir o arquivo.
Dai o codigo que vem depois (os agentes, as consultas, as verificacoes) usa esses nomes de coluna diretamente. Se eu escrever gdf["ZONA"] e o nome real for gdf["SIGLA"], o sistema inteiro quebra. E aconteceu bastante ate entendermos essa parte.

In [61]:
import geopandas as gpd

caminhos = {
    "zoneamento":  "/content/bhgeo/ZONEAMENTO_11181.shp",
    "ade":         "/content/bhgeo/ADE_11181.shp",
    "ade_setores": "/content/bhgeo/ADE_SETORES_11181.shp",
    "viario":      "/content/bhgeo/CLASSIFICACAO_VIARIA_11181.shp",
}

gdfs = {}
for nome, caminho in caminhos.items():
    print(f"\n{'='*55}")
    print(f"Camada: {nome}")
    try:
        gdf = gpd.read_file(caminho).to_crs(epsg=4326)
        gdfs[nome] = gdf
        print(f"✅ {len(gdf)} registros carregados")
        print(f"Colunas: {list(gdf.columns)}")
        print(f"Primeiro registro:")
        print(gdf.drop(columns="geometry").iloc[0].to_dict())
    except Exception as e:
        print(f"❌ Erro: {e}")


Camada: zoneamento
✅ 1716 registros carregados
Colunas: ['ID_ZONEAME', 'DESC_TIPO_', 'SIGLA_TIPO', 'geometry']
Primeiro registro:
{'ID_ZONEAME': 34.0, 'DESC_TIPO_': 'Area Especial de Interesse Social -1', 'SIGLA_TIPO': 'AEIS_1'}

Camada: ade
✅ 25 registros carregados
Colunas: ['ID_ADE', 'NOME_ADE', 'geometry']
Primeiro registro:
{'ID_ADE': 9.0, 'NOME_ADE': 'ADE Estoril'}

Camada: ade_setores
✅ 75 registros carregados
Colunas: ['ID_SETOR_A', 'NOME_SETOR', 'DESC_USO_S', 'geometry']
Primeiro registro:
{'ID_SETOR_A': 50.0, 'NOME_SETOR': 'ADE Mirantes - PALMARES - Setor 1', 'DESC_USO_S': None}

Camada: viario
✅ 54492 registros carregados
Colunas: ['ID_CLASSIF', 'TP_LOG', 'NO_LOG', 'CLASSIFICA', 'SUBDIVISAO', 'AFASTAMENT', 'TIPO_LARGU', 'DESCRICAO_', 'geometry']
Primeiro registro:
{'ID_CLASSIF': 271.0, 'TP_LOG': 'BEC', 'NO_LOG': 'UM MIL QUATROCENTOS E SETENTA E QUATRO', 'CLASSIFICA': 'LOCAL', 'SUBDIVISAO': None, 'AFASTAMENT': '0', 'TIPO_LARGU': 'A', 'DESCRICAO_': 'LARGURA DA VIA < 10 m'}


8. TypedDict

Aqui a gente cria uma especie de "pasta compartilhada" para que cada agente vem pega o que precisa, analisa e depois passa pro proximo dar continuidade na task.
Ele e um "contrato de dados" (Estado Conformidade) compartilhado entre todos os agentes do grafo. Cada campo representa uma informacao que um agente produz e outro pode consumir. Campos Optional comecam como none e sao preenchidos progressivamente conforme o grafo avanca de no em no.

In [62]:
from typing import TypedDict, Optional

class EstadoConformidade(TypedDict):
    # Input
    caminho_ifc:        str
    pergunta_usuario:   str
    # Extraido do IFC
    uso_edificacao:     Optional[str]
    num_pavimentos:     Optional[int]
    altura_total_m:     Optional[float]
    latitude:           Optional[float]
    longitude:          Optional[float]
    area_lote_m2:       Optional[float]
    area_construida_m2: Optional[float]
    ca_projetado:       Optional[float]
    # Verificacao de uso
    zona_sigla:         Optional[str]
    zona_descricao:     Optional[str]
    uso_permitido:      Optional[bool]
    resultado_uso:      Optional[str]
    # Verificacao de gabarito
    via_classificacao:  Optional[str]
    via_nome:           Optional[str]
    gabarito_maximo:    Optional[int]
    gabarito_ok:        Optional[bool]
    resultado_gabarito: Optional[str]
    # Verificacao de ADE
    ade_nome:           Optional[str]
    ade_setor:          Optional[str]
    ade_uso_descricao:  Optional[str]
    ca_maximo_ade:      Optional[float]
    ade_ok:             Optional[bool]
    resultado_ade:      Optional[str]
    # Saida final
    laudo_final:        Optional[str]
    conflitos_total:    Optional[int]

print("✅ Estado definido")

✅ Estado definido


8. Carregamento dos Shapefiles e funcoes de consulta

Aqui e onde entram os dados urbanos que o agente vai consultar.

Uma coisa importante, o agente usa dois tipos de dados aqui:

Os Shapefiles respondem ONDE o lote esta: em qual zona, se tem ADE, qual o tipo de via e isso vem direto dos dados reais da prefeitura.

ja o PARAMETROS_ZONA, responde O QUE E PERMITIDO naquela zona: gabarito, usos, CA. Esses valores eu extrai manualmente da Lei 11.181/2019 e coloquei no codigo igual voce fez com os valores da NBR na aula.

As 3 funcoes que os agentes chamam pra consultar os dados:

consultar_zona(): em qual zona urbanistica esta o lote?
consultar_ade(): tem ADE incidindo nesse ponto?
consultar_via(): qual a classificacao da via mais proxima?

In [63]:
# Lendo os 4 Shapefiles
gdf_zona = gpd.read_file("/content/bhgeo/ZONEAMENTO_11181.shp").to_crs(epsg=4326)
gdf_ade  = gpd.read_file("/content/bhgeo/ADE_11181.shp").to_crs(epsg=4326)
gdf_ade_setores = gpd.read_file("/content/bhgeo/ADE_SETORES_11181.shp").to_crs(epsg=4326)
gdf_vias = gpd.read_file("/content/bhgeo/CLASSIFICACAO_VIARIA_11181.shp").to_crs(epsg=4326)

# Entramos com os parametros urbanisticos da lei 11.181/2019
PARAMETROS_ZONA = {
    "ZOP":    {"usos_permitidos": ["RESIDENTIAL", "COMMERCIAL", "OFFICE"],
               "gabarito_via_coletora": 6, "gabarito_via_local": 4, "ca_max": 2.0},
    "ZAM":    {"usos_permitidos": ["RESIDENTIAL"],
               "gabarito_via_coletora": 4, "gabarito_via_local": 3, "ca_max": 1.5},
    "ZPAM":   {"usos_permitidos": [],
               "gabarito_via_coletora": 0, "gabarito_via_local": 0, "ca_max": 0.0},
    "ZEIS":   {"usos_permitidos": ["RESIDENTIAL"],
               "gabarito_via_coletora": 4, "gabarito_via_local": 3, "ca_max": 1.2},
    "AEIS_1": {"usos_permitidos": ["RESIDENTIAL"],
               "gabarito_via_coletora": 4, "gabarito_via_local": 3, "ca_max": 1.2},
    "OP-3": {"usos_permitidos": ["RESIDENTIAL", "COMMERCIAL", "OFFICE"], "gabarito_via_coletora": 6, "gabarito_via_local": 4, "gabarito_via_arterial": 4, "ca_max": 2.0,},
}

# Funcoes de consulta para os agentes usar os nomes de coluna reais da PBH
def consultar_zona(lat, lon):
    ponto = Point(lon, lat)
    resultado = gdf_zona[gdf_zona.geometry.contains(ponto)]
    if resultado.empty:
        return {"sigla": "NAO_ENCONTRADA", "descricao": "Fora do perímetro mapeado"}
    row = resultado.iloc[0]
    return {"sigla": row["SIGLA_TIPO"], "descricao": row["DESC_TIPO_"]}

def consultar_ade(lat, lon):
    ponto = Point(lon, lat)
    ade_geral = gdf_ade[gdf_ade.geometry.contains(ponto)]
    ade_setor = gdf_ade_setores[gdf_ade_setores.geometry.contains(ponto)]
    nome_ade   = ade_geral.iloc[0]["NOME_ADE"] if not ade_geral.empty else None
    nome_setor = ade_setor.iloc[0]["NOME_SETOR"] if not ade_setor.empty else None
    desc_uso   = ade_setor.iloc[0]["DESC_USO_S"] if not ade_setor.empty else None
    return {"nome_ade": nome_ade, "nome_setor": nome_setor, "desc_uso": desc_uso}

def consultar_via(lat, lon):
    ponto  = Point(lon, lat)
    buffer = ponto.buffer(0.001)
    proximas = gdf_vias[gdf_vias.geometry.intersects(buffer)]
    if proximas.empty:
        return {"classificacao": "LOCAL", "nome": "Não identificada"}
    row = proximas.iloc[0]
    return {"classificacao": row["CLASSIFICA"], "nome": row["NO_LOG"]}

print("✅ Shapefiles prontos e funções de consulta prontas")
print(f"   Zonas:    {len(gdf_zona)} polígonos")
print(f"   ADEs:     {len(gdf_ade)} polígonos")
print(f"   Setores:  {len(gdf_ade_setores)} polígonos")
print(f"   Vias:     {len(gdf_vias)} registros")

✅ Shapefiles prontos e funções de consulta prontas
   Zonas:    1716 polígonos
   ADEs:     25 polígonos
   Setores:  75 polígonos
   Vias:     54492 registros


10. Nos do LangGraph

Aqui e onde definimos os 5 agentes do sistema. Cada no e uma funcao Python que recebe o estado compartilhado, faz sua parte e devolve so os campos que sao de responsabilidade dele.

Definimos os seguintes nos usando o fluxo condicional:

No 1 - Extrator IFC: abre o .ifc com o IfcOpenShell e extrai tudo que o sistema precisa: uso, pavimentos, coordenadas e areas.

No 2 - Verificador de Uso: pega as coordenadas que o No 1 extraiu, consulta o Shapefile de zoneamento e verifica se o uso do projeto e permitido naquela zona.

No 3 - Verificador de Gabarito: consulta a classificacao viaria e verifica se o numero de pavimentos respeita o limite da zona.

No 4 - Verificador de ADE: verifica se existe alguma Area de Diretriz Especial incidindo sobre o lote e qual o setor.

No 5 - Sintetizador: pega o resultado de todas as verificacoes anteriores e gera o laudo final em linguagem natural usando o Claude.

In [64]:
def no_extrator_ifc(estado: EstadoConformidade) -> dict:
    print("📐 [Nó 1 - Extrator IFC] Lendo modelo...")
    modelo = ifcopenshell.open(estado["caminho_ifc"])

    sites = modelo.by_type("IfcSite")
    lat = lon = area_lote = None
    if sites:
        site = sites[0]
        if site.RefLatitude and site.RefLongitude:
            def dms(v):
                return abs(v[0]) + v[1]/60 + v[2]/3600
            lat = -dms(site.RefLatitude)
            lon = -dms(site.RefLongitude)
        props = ifc_util.get_psets(site)
        for pset in props.values():
            if "GrossAreaPlot" in pset:
                area_lote = float(pset["GrossAreaPlot"])

    buildings = modelo.by_type("IfcBuilding")
    uso = "NAO_DECLARADO"
    if buildings:
        obj_type = getattr(buildings[0], "ObjectType", None)
        if obj_type:
            uso = obj_type
        else:
            nome = getattr(buildings[0], "Name", "") or ""
            nome_lower = nome.lower()
            if "unifamiliar" in nome_lower or "residencial" in nome_lower or "residencia" in nome_lower:
                uso = "RESIDENTIAL"
            elif "comercial" in nome_lower:
                uso = "COMMERCIAL"
            elif "escritorio" in nome_lower or "office" in nome_lower:
                uso = "OFFICE"
            else:
                uso = "RESIDENTIAL"

    pavs  = modelo.by_type("IfcBuildingStorey")
    cotas = [float(p.Elevation) for p in pavs if p.Elevation is not None]
    if cotas and max(cotas) < 100:
        altura = round(max(cotas) - min(cotas), 1) if len(cotas) > 1 else len(pavs) * 3.0
    else:
        altura = round((max(cotas) - min(cotas)) / 1000, 1) if len(cotas) > 1 else len(pavs) * 3.0

    espacos = modelo.by_type("IfcSpace")
    area_construida = 0.0
    for e in espacos:
        for pset in ifc_util.get_psets(e).values():
            if "GrossFloorArea" in pset:
                area_construida += float(pset["GrossFloorArea"])
    ca = round(area_construida / area_lote, 2) if area_lote else None

    print("   Uso: " + str(uso) + " | Pavimentos: " + str(len(pavs)) + " | Altura: " + str(altura) + "m")
    print("   Coordenadas: " + str(lat) + ", " + str(lon))

    return {
        "uso_edificacao":     uso,
        "num_pavimentos":     len(pavs),
        "altura_total_m":     altura,
        "latitude":           lat,
        "longitude":          lon,
        "area_lote_m2":       area_lote,
        "area_construida_m2": round(area_construida, 1),
        "ca_projetado":       ca,
    }


def no_verificador_uso(estado: EstadoConformidade) -> dict:
    print("🏙️ [Nó 2 - Verificador de Uso] Consultando zoneamento...")

    zona    = consultar_zona(estado["latitude"], estado["longitude"])
    sigla   = zona["sigla"]
    params  = PARAMETROS_ZONA.get(sigla, {})
    usos_ok = params.get("usos_permitidos", [])
    uso     = estado["uso_edificacao"]
    ok      = uso in usos_ok

    conteudo = "\n".join([
        "Uso declarado no IFC: " + str(uso),
        "Zona: " + str(sigla) + " (" + str(zona["descricao"]) + ")",
        "Usos permitidos nessa zona: " + str(usos_ok),
        "Uso permitido: " + str(ok),
        "De o veredicto tecnico.",
    ])

    resposta = llm.invoke([
        SystemMessage(content="Especialista em legislacao urbanistica de BH. Responda em 2 linhas objetivas. Sem assinatura."),
        HumanMessage(content=conteudo)
    ])

    print("   Zona: " + str(sigla) + " | Uso permitido: " + str(ok))
    return {
        "zona_sigla":     sigla,
        "zona_descricao": zona["descricao"],
        "uso_permitido":  ok,
        "resultado_uso":  resposta.content,
    }


def no_verificador_gabarito(estado: EstadoConformidade) -> dict:
    print("📏 [Nó 3 - Verificador de Gabarito] Verificando altura...")

    via     = consultar_via(estado["latitude"], estado["longitude"])
    params  = PARAMETROS_ZONA.get(estado["zona_sigla"], {})
    chave   = "gabarito_via_coletora" if "COLETORA" in via["classificacao"].upper() else "gabarito_via_local"
    gab_max = params.get(chave, 4)
    ok      = estado["num_pavimentos"] <= gab_max

    conteudo = "\n".join([
        "Pavimentos projetados: " + str(estado["num_pavimentos"]) + " (" + str(estado["altura_total_m"]) + "m)",
        "Zona: " + str(estado["zona_sigla"]),
        "Via: " + str(via["nome"]) + " (Classificacao: " + str(via["classificacao"]) + ")",
        "Gabarito maximo permitido: " + str(gab_max) + " pavimentos",
        "Conforme: " + str(ok),
        "De o veredicto tecnico.",
    ])

    resposta = llm.invoke([
        SystemMessage(content="Especialista em legislacao urbanistica de BH. Responda em 2 linhas objetivas. Sem assinatura."),
        HumanMessage(content=conteudo)
    ])

    print("   Via: " + str(via["classificacao"]) + " | Gabarito max: " + str(gab_max) + " | OK: " + str(ok))
    return {
        "via_classificacao":  via["classificacao"],
        "via_nome":           via["nome"],
        "gabarito_maximo":    gab_max,
        "gabarito_ok":        ok,
        "resultado_gabarito": resposta.content,
    }


def no_verificador_ade(estado: EstadoConformidade) -> dict:
    print("🗺️ [Nó 4 - Verificador ADE] Verificando sobreposicoes...")

    ade = consultar_ade(estado["latitude"], estado["longitude"])
    ca  = estado.get("ca_projetado")

    conteudo = "\n".join([
        "ADE identificada: " + str(ade["nome_ade"] or "Nenhuma"),
        "Setor: " + str(ade["nome_setor"] or "Nenhum"),
        "Descricao de uso: " + str(ade["desc_uso"] or "Nao especificada"),
        "CA projetado: " + str(ca),
        "Ha conflito com a ADE? De o veredicto.",
    ])

    resposta = llm.invoke([
        SystemMessage(content="Especialista em legislacao urbanistica de BH. Responda em 2 linhas objetivas. Sem assinatura."),
        HumanMessage(content=conteudo)
    ])

    print("   ADE: " + str(ade["nome_ade"] or "Nenhuma") + " | Setor: " + str(ade["nome_setor"] or "Nenhum"))
    return {
        "ade_nome":          ade["nome_ade"],
        "ade_setor":         ade["nome_setor"],
        "ade_uso_descricao": ade["desc_uso"],
        "ade_ok":            True,
        "resultado_ade":     resposta.content,
    }


def no_sintetizador(estado: EstadoConformidade) -> dict:
    print("📋 [Nó 5 - Sintetizador] Gerando laudo final...")

    conflitos = sum([
        not estado.get("uso_permitido", True),
        not estado.get("gabarito_ok",   True),
        not estado.get("ade_ok",        True),
    ])

    parte1 = "Pergunta do usuario: " + str(estado["pergunta_usuario"])
    parte2 = "VERIFICACAO 1 - USO DO SOLO:\n" + str(estado.get("resultado_uso"))
    parte3 = "VERIFICACAO 2 - GABARITO:\n" + str(estado.get("resultado_gabarito"))
    parte4 = "VERIFICACAO 3 - ADE:\n" + str(estado.get("resultado_ade"))
    parte5 = "Total de conflitos: " + str(conflitos)
    parte6 = "Responda a pergunta com base nas verificacoes acima."

    conteudo = "\n\n".join([parte1, parte2, parte3, parte4, parte5, parte6])

    resposta = llm.invoke([
        SystemMessage(content=(
            "Voce e um sistema automatizado de analise urbanistica de Belo Horizonte. "
            "Gere um laudo tecnico direto e objetivo. "
            "Maximo 150 palavras. "
            "Assine como: uRBAN , Agente de Conformidade Urbanística."
            "Inclua uma breve saudacao ou despedida."
        )),
        HumanMessage(content=conteudo)
    ])

    return {
        "laudo_final":     resposta.content,
        "conflitos_total": conflitos,
    }


print("✅ 5 nos definidos:")
print("   No 1:", no_extrator_ifc.__name__)
print("   No 2:", no_verificador_uso.__name__)
print("   No 3:", no_verificador_gabarito.__name__)
print("   No 4:", no_verificador_ade.__name__)
print("   No 5:", no_sintetizador.__name__)

✅ 5 nos definidos:
   No 1: no_extrator_ifc
   No 2: no_verificador_uso
   No 3: no_verificador_gabarito
   No 4: no_verificador_ade
   No 5: no_sintetizador


11. Montagem do grafo LangGraph com condicionais

Aqui nos conectamos os nos e defino como o sistema vai tomar decisoes. Nao e um fluxo fixo - o grafo decide qual caminho seguir dependendo do que cada agente encontrou.
Colocamos duas decisoes condicionais:
A primeira e depois do verificador de uso: se o uso nao for permitido na zona o sistema ja encerra e emite um alerta critico. Se for permitido continua pra verificar o gabarito.
A segunda e depois do verificador de gabarito: se o gabarito estiver acima do limite o sistema encerra e emite um alerta de gabarito. Se estiver ok continua
pra verificar a ADE.
Fizemos como o roteamento inteligente da aula assim o grafo nao percorre sempre o mesmo caminho, ele decide baseado nos dados que os agentes encontraram.

In [65]:
# Montagem do LangGraph com transições condicionais

def roteador_uso(estado: EstadoConformidade) -> str:
    """Decide o próximo nó após verificar o uso do solo."""
    if estado.get("uso_permitido") == False:
        print("⚠️  [Roteador] Uso NÃO permitido → acionando alerta crítico")
        return "alerta_critico"
    else:
        print("✅ [Roteador] Uso permitido → continuando verificação")
        return "verificador_gabarito"

def roteador_gabarito(estado: EstadoConformidade) -> str:
    """Decide o próximo nó após verificar o gabarito."""
    if estado.get("gabarito_ok") == False:
        print("⚠️  [Roteador] Gabarito REPROVADO → acionando alerta de gabarito")
        return "alerta_gabarito"
    else:
        print("✅ [Roteador] Gabarito aprovado → continuando para ADE")
        return "verificador_ade"

def no_alerta_critico(estado: EstadoConformidade) -> dict:
    """Nó ativado quando o uso é incompatível com a zona."""
    print("🚨 [Alerta Crítico] Uso incompatível com a zona!")
    msg = (
        "ALERTA CRÍTICO: O uso declarado no projeto é INCOMPATÍVEL "
        "com a zona urbanística do lote. "
        "Zona: " + str(estado.get("zona_sigla")) + " | "
        "Uso: " + str(estado.get("uso_edificacao")) + ". "
        "O projeto não pode ser aprovado sem alteração do uso ou mudança de zona."
    )
    return {
        "laudo_final":     msg,
        "conflitos_total": 1,
    }

def no_alerta_gabarito(estado: EstadoConformidade) -> dict:
    """Nó ativado quando o gabarito está acima do permitido."""
    print("🚨 [Alerta Gabarito] Gabarito acima do limite!")
    msg = (
        "ALERTA DE GABARITO: O número de pavimentos excede o limite permitido. "
        "Projetado: " + str(estado.get("num_pavimentos")) + " pavimentos | "
        "Máximo permitido: " + str(estado.get("gabarito_maximo")) + " pavimentos. "
        "Zona: " + str(estado.get("zona_sigla")) + " | "
        "Via: " + str(estado.get("via_classificacao")) + ". "
        "Necessária redução do gabarito para aprovação do projeto."
    )
    return {
        "laudo_final":     msg,
        "conflitos_total": 1,
    }


# Monta o grafo com roteamento condicional

builder = StateGraph(EstadoConformidade)

# Registra todos os nós
builder.add_node("extrator_ifc",         no_extrator_ifc)
builder.add_node("verificador_uso",      no_verificador_uso)
builder.add_node("verificador_gabarito", no_verificador_gabarito)
builder.add_node("verificador_ade",      no_verificador_ade)
builder.add_node("sintetizador",         no_sintetizador)
builder.add_node("alerta_critico",       no_alerta_critico)
builder.add_node("alerta_gabarito",      no_alerta_gabarito)

# Arestas fixas
builder.add_edge(START,              "extrator_ifc")
builder.add_edge("extrator_ifc",     "verificador_uso")

# Aresta condicional 1 — após verificar uso
builder.add_conditional_edges(
    "verificador_uso",
    roteador_uso,
    {
        "verificador_gabarito": "verificador_gabarito",
        "alerta_critico":       "alerta_critico",
    }
)

# Aresta condicional 2 — após verificar gabarito
builder.add_conditional_edges(
    "verificador_gabarito",
    roteador_gabarito,
    {
        "verificador_ade":  "verificador_ade",
        "alerta_gabarito":  "alerta_gabarito",
    }
)

# Arestas fixas finais
builder.add_edge("verificador_ade",  "sintetizador")
builder.add_edge("alerta_critico",   END)
builder.add_edge("alerta_gabarito",  END)
builder.add_edge("sintetizador",     END)

sistema = builder.compile()

print("✅ Grafo compilado com transições condicionais")
print("   Fluxo: extrator → uso →[condicional]→ gabarito →[condicional]→ ADE → sintetizador")
print("   Atalhos: uso inválido → alerta_critico | gabarito inválido → alerta_gabarito")

✅ Grafo compilado com transições condicionais
   Fluxo: extrator → uso →[condicional]→ gabarito →[condicional]→ ADE → sintetizador
   Atalhos: uso inválido → alerta_critico | gabarito inválido → alerta_gabarito


12. Interface de perguntas livres

AQUI VIMOS QUE A COISA FUNCIONA MESMO! S2

Criamos essa celula para testar o agente. Voce digita qualquer pergunta sobre o projeto em linguagem natural e o sistema roda o grafo completo, ele le o IFC, consulta os Shapefiles da PBH e devolve o laudo gerado pelo Claude.
O resultado fica salvo na variavel resultado pra poder exportar na celula seguinte.

In [66]:
pergunta = input("Digite sua pergunta sobre o projeto: ")

print("\n" + "="*65)
print("PERGUNTA: " + pergunta)
print("="*65)

resultado = sistema.invoke({
    "caminho_ifc": "/content/espirito_santo_507.ifc",
    "pergunta_usuario": pergunta,
})

print("\n📄 LAUDO FINAL:")
print(resultado["laudo_final"])
print("\n⚠️  Conflitos detectados: " + str(resultado["conflitos_total"]))


Digite sua pergunta sobre o projeto: O tipo de uso do projeto esta de acordo com a lei de zoneamento local?

PERGUNTA: O tipo de uso do projeto esta de acordo com a lei de zoneamento local?
📐 [Nó 1 - Extrator IFC] Lendo modelo...
   Uso: RESIDENTIAL | Pavimentos: 2 | Altura: 3.0m
   Coordenadas: -19.92777777777778, -43.93944444444444
🏙️ [Nó 2 - Verificador de Uso] Consultando zoneamento...
   Zona: OP-3 | Uso permitido: True
✅ [Roteador] Uso permitido → continuando verificação
📏 [Nó 3 - Verificador de Gabarito] Verificando altura...
   Via: ARTERIAL | Gabarito max: 4 | OK: True
✅ [Roteador] Gabarito aprovado → continuando para ADE
🗺️ [Nó 4 - Verificador ADE] Verificando sobreposicoes...
   ADE: ADE Avenida do Contorno | Setor: ADE Avenida do Contorno - Setor ADE Rua da Bahia Viva
📋 [Nó 5 - Sintetizador] Gerando laudo final...

📄 LAUDO FINAL:
Prezado usuário,

Com base na análise técnica realizada, **SIM, o tipo de uso do projeto está de acordo com a lei de zoneamento local**.

O uso re

13. Exportacao do resultado como log estruturado

Essa celula pega o ultimo resultado que o sistema gerou e salva como arquivo .txt no formato log e la tem tudo organizado: data e informacoes do arquivo IFC, o que foi extraido do modelo, o resultado de cada verificacao de uso, gabarito e ADE, e o laudo final completo.
E so executar essa celula depois de fazer qualquer pergunta que o arquivo e gerado e baixado automaticamente ;)

In [ ]:
from datetime import datetime
from google.colab import files
import ifcopenshell
import re

def baixar_log():

    # Pega o nome do site direto do IFC para usar no nome do arquivo
    modelo    = ifcopenshell.open("/content/espirito_santo_507.ifc")
    sites     = modelo.by_type("IfcSite")
    nome_site = sites[0].Name if sites else "projeto"

    # Limpa o nome para usar como nome de arquivo
    # Remove caracteres especiais e espacos
    nome_limpo = re.sub(r'[^a-zA-Z0-9]', '_', nome_site)
    nome_limpo = re.sub(r'_+', '_', nome_limpo).strip('_').lower()
    nome_arquivo = "log_" + nome_limpo + ".txt"

    linhas = [
        "=" * 60,
        "SISTEMA AGENTIC — CONFORMIDADE URBANÍSTICA BH",
        "=" * 60,
        "Data:          " + datetime.now().strftime("%d/%m/%Y %H:%M"),
        "Arquivo IFC:   espirito_santo_507.ifc",
        "Site:          " + nome_site,
        "Coordenadas:   -19.9278, -43.9394",
        "Dados urbanos: Shapefiles PBH — Lei 11.181/2019",
        "-" * 60,
        "",
        "PERGUNTA DO USUÁRIO:",
        "  " + str(resultado.get("pergunta_usuario", "-")),
        "",
        "DADOS EXTRAÍDOS DO IFC:",
        "  Uso declarado:    " + str(resultado.get("uso_edificacao", "-")),
        "  Pavimentos:       " + str(resultado.get("num_pavimentos", "-")),
        "  Altura total:     " + str(resultado.get("altura_total_m", "-")) + "m",
        "  Área construída:  " + str(resultado.get("area_construida_m2", "-")) + " m²",
        "  CA projetado:     " + str(resultado.get("ca_projetado", "-")),
        "",
        "VERIFICAÇÕES:",
        "  Zona:              " + str(resultado.get("zona_sigla", "-")),
        "  Zona descrição:    " + str(resultado.get("zona_descricao", "-")),
        "  Uso permitido:     " + ("✅ Sim" if resultado.get("uso_permitido") else "❌ Não"),
        "  Via:               " + str(resultado.get("via_classificacao", "-")),
        "  Gabarito máximo:   " + str(resultado.get("gabarito_maximo", "-")) + " pavimentos",
        "  Gabarito conforme: " + ("✅ Sim" if resultado.get("gabarito_ok") else "❌ Não"),
        "  ADE identificada:  " + str(resultado.get("ade_nome") or "Nenhuma"),
        "  Setor ADE:         " + str(resultado.get("ade_setor") or "-"),
        "  Conflitos:         " + str(resultado.get("conflitos_total", 0)),
        "",
        "LAUDO FINAL:",
    ]

    for linha in str(resultado.get("laudo_final", "-")).split("\n"):
        linhas.append("  " + linha)

    linhas += [
        "",
        "=" * 60,
        "Gerado por: Sistema Multi-Agente LangGraph",
        "Framework:  LangGraph + IfcOpenShell + GeoPandas",
        "LLM:        Claude Sonnet (Anthropic)",
        "=" * 60,
    ]

    with open(nome_arquivo, "w", encoding="utf-8") as f:
        f.write("\n".join(linhas))

    print("✅ Log gerado: " + nome_arquivo)
    files.download(nome_arquivo)

baixar_log()

✅ Log gerado: log_rua_espirito_santo_507_belo_horizonte_mg.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>